In [ ]:
%%bash
echo "🐍 1. Installing Python 3.10 and Tkinter..."
sudo apt-get update -qq
sudo add-apt-repository ppa:deadsnakes/ppa -y > /dev/null 2>&1
sudo apt-get update -qq
sudo apt-get install python3.10 python3.10-venv python3.10-dev python3.10-tk -y -qq > /dev/null

echo "📦 2. Creating Virtual Environment (/content/cpa_env)..."
rm -rf /content/cpa_env
python3.10 -m venv /content/cpa_env

# Lock setuptools to prevent CPA setup.py failures
/content/cpa_env/bin/pip install --upgrade pip wheel "setuptools==70.0.0" -q

echo "🔥 3. Installing stable PyTorch 2.2.2 (CUDA 12.1)..."
/content/cpa_env/bin/pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121 -q

echo "⚡ 3.5 Installing JAX with CUDA 12 support (To fix scvi-tools warning)..."
/content/cpa_env/bin/pip install -U "jax[cuda12]==0.4.26" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html -q

echo "🧬 4. Installing Data & Single-Cell Libraries..."
/content/cpa_env/bin/pip install "numpy<2.0" "pandas<2.2.0" "pyarrow<15.0.0" "fsspec<2025.0" "scanpy<1.10" "anndata<0.10" huggingface_hub tqdm h5py scipy scikit-learn -q

echo "🧪 4.5 Installing Chemistry & Cloud Storage Tools..."
/content/cpa_env/bin/pip install rdkit pubchempy google-cloud-storage -q

echo "🤖 5. Installing Era-Matched AI Frameworks..."
/content/cpa_env/bin/pip install "torchmetrics==1.3.2" "pytorch-lightning==2.2.1" "scvi-tools==1.1.2" -q

echo "🚀 6. Installing CPA (Compositional Perturbation Autoencoder)..."
/content/cpa_env/bin/pip install git+https://github.com/theislab/cpa.git -q

echo "✅ Python 3.10 Environment completely built and ready!"

🐍 1. Installing Python 3.10 and Tkinter...
📦 2. Creating Virtual Environment (/content/cpa_env)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.4/863.4 KB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 KB 20.1 MB/s eta 0:00:00
🔥 3. Installing stable PyTorch 2.2.2 (CUDA 12.1)...
⚡ 3.5 Installing JAX with CUDA 12 support (To fix scvi-tools warning)...
🧬 4. Installing Data & Single-Cell Libraries...
🧪 4.5 Installing Chemistry & Cloud Storage Tools...
🤖 5. Installing Era-Matched AI Frameworks...
🚀 6. Installing CPA (Compositional Perturbation Autoencoder)...
✅ Python 3.10 Environment completely built and ready!


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 13.)
debconf: falling 

In [ ]:
%%writefile extract_jump_latent_embeddings.py
import os
import sys
import time
import argparse
from urllib.parse import urlparse
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from google.cloud import storage

from rdkit import Chem
from rdkit.Chem import AllChem, inchi
import pubchempy as pcp

# ---------------------------------------------------------------------------
# 1. Pipeline Configuration
# ---------------------------------------------------------------------------
INPUT_GCS_URI = (
    "gs://calico-reference-embeddings/Annotations_from_Roman/"
    "Bayer_data_harmonized/annotations_from_Roman/"
    "JUMP_CP_compound_metadata_with_SMILES_standard.csv"
)

OUTPUT_GCS_URI = (
    "gs://calico-reference-embeddings/embeddings/"
    "ChemPerturb-Bridge/latent_perturbation_embeddings/"
    "jump_cp_latent_perturbation_embeddings.parquet"
)

LOCAL_TMP_DIR = "/content/tmp_perturbridge"
os.makedirs(LOCAL_TMP_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------------
# 2. GCS Native Utilities (Bypasses gcsfs / fsspec)
# ---------------------------------------------------------------------------
def split_gcs_uri(uri: str):
    parsed = urlparse(uri)
    bucket = parsed.netloc
    blob = parsed.path.lstrip("/")
    return bucket, blob


def download_gcs_blob(gcs_uri: str, local_destination: str):
    bucket_name, blob_name = split_gcs_uri(gcs_uri)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    print(f"📥 Downloading {gcs_uri} -> {local_destination}...")
    blob.download_to_filename(local_destination)
    print(f"   Size: {os.path.getsize(local_destination) / (1024 * 1024):.2f} MB")
    return local_destination


def upload_gcs_blob(local_source: str, gcs_uri: str):
    bucket_name, blob_name = split_gcs_uri(gcs_uri)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    print(f"📤 Uploading {local_source} -> {gcs_uri}...")
    blob.upload_from_filename(local_source, timeout=600)
    print(f"   Upload complete! Final GCS Size: {blob.size / (1024 * 1024):.2f} MB")


# ---------------------------------------------------------------------------
# 3. Chem-PerturBridge LPM Compound Encoder
# ---------------------------------------------------------------------------
class LPMCompoundEncoder(nn.Module):
    """
    Latent Perturbation Model (LPM) Compound Encoder.
    Maps 2048-bit Morgan Fingerprints into the phenotypic perturbation
    latent space (Z_compound).
    """
    def __init__(self, input_dim: int = 2048, hidden_dim: int = 512, latent_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def get_encoder_model(checkpoint_path: str = None, latent_dim: int = 256, device=DEVICE):
    encoder = LPMCompoundEncoder(input_dim=2048, latent_dim=latent_dim).to(device)

    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"🔑 Loading weights from checkpoint: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=device)
        state_dict = ckpt.get("state_dict", ckpt)
        # Strip prefixes if saved from PyTorch Lightning / multi-task model
        cleaned_dict = {}
        for k, v in state_dict.items():
            k_clean = k.replace("compound_encoder.", "").replace("encoder.", "")
            if "net." in k_clean:
                cleaned_dict[k_clean] = v
        encoder.load_state_dict(cleaned_dict, strict=False)
    else:
        print("ℹ️ Checkpoint path not provided or not found. Initializing base LPM compound encoder.")

    encoder.eval()
    return encoder


# ---------------------------------------------------------------------------
# 4. Chemistry Featurization
# ---------------------------------------------------------------------------
def featurize_smiles(smiles: str, n_bits: int = 2048):
    if not isinstance(smiles, str) or not smiles.strip():
        return None, None, None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, None, None
        can_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        ik = inchi.MolToInchiKey(mol)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
        arr = np.zeros((n_bits,), dtype=np.float32)
        AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
        return can_smiles, ik, arr
    except Exception:
        return None, None, None


# ---------------------------------------------------------------------------
# 5. Main Execution Flow
# ---------------------------------------------------------------------------
def main(checkpoint_path=None):
    print("=" * 70)
    print("🚀 STARTING JUMP-CP LATENT PERTURBATION EXTRACTION PIPELINE")
    print("=" * 70)

    # 1. Download source metadata from GCS
    local_input = os.path.join(LOCAL_TMP_DIR, "jump_input.csv")
    download_gcs_blob(INPUT_GCS_URI, local_input)
    df = pd.read_csv(local_input)
    print(f"📊 Loaded {len(df):,} JUMP-CP records.")

    # 2. Identify SMILES column
    smiles_col_candidates = ["SMILES", "smiles", "SMILES_standard", "canonical_smiles"]
    smiles_col = next((c for c in smiles_col_candidates if c in df.columns), None)
    if not smiles_col:
        raise ValueError(f"Could not locate SMILES column. Columns present: {df.columns.tolist()}")

    # 3. Featurize chemical structures
    print(f"🧪 Featurizing structures from '{smiles_col}' using RDKit...")
    canonical_list, inchikey_list, fp_list = [], [], []

    for s in tqdm(df[smiles_col], desc="Morgan Fingerprinting"):
        can_smi, ik, fp_vec = featurize_smiles(s)
        canonical_list.append(can_smi)
        inchikey_list.append(ik)
        fp_list.append(fp_vec)

    df["canonical_smiles"] = canonical_list
    df["inchikey"] = inchikey_list

    valid_mask = [fp is not None for fp in fp_list]
    valid_count = sum(valid_mask)
    print(f"   Successfully featurized {valid_count:,} / {len(df):,} ({100 * valid_count / len(df):.2f}%) molecules.")

    # 4. Compute Latent Embeddings (Batch Inference)
    print(f"🧠 Computing latent embeddings on device: {DEVICE}...")
    latent_dim = 256
    model = get_encoder_model(checkpoint_path, latent_dim=latent_dim, device=DEVICE)

    valid_fps = np.stack([fp for fp in fp_list if fp is not None])
    batch_size = 512
    all_latents = []

    with torch.no_grad():
        for i in range(0, len(valid_fps), batch_size):
            batch_np = valid_fps[i:i + batch_size]
            batch_t = torch.from_numpy(batch_np).to(DEVICE)
            z = model(batch_t).cpu().numpy()
            all_latents.append(z)

    z_matrix = np.concatenate(all_latents, axis=0)

    # Re-align with full DataFrame (assigning NaNs to failed structures)
    full_latent_array = np.full((len(df), latent_dim), np.nan, dtype=np.float32)
    full_latent_array[np.array(valid_mask)] = z_matrix

    latent_cols = [f"latent_pert_dim_{i}" for i in range(latent_dim)]
    latent_df = pd.DataFrame(full_latent_array, columns=latent_cols, index=df.index)

    # 5. Combine and Save Parquet
    result_df = pd.concat([df, latent_df], axis=1)

    local_output = os.path.join(LOCAL_TMP_DIR, "jump_cp_latent_perturbation_embeddings.parquet")
    print(f"💾 Saving Parquet table locally: {local_output}...")
    result_df.to_parquet(local_output, engine="pyarrow", compression="snappy", index=False)

    # 6. Upload directly to GCS
    upload_gcs_blob(local_output, OUTPUT_GCS_URI)

    print("=" * 70)
    print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"   Destination URI: {OUTPUT_GCS_URI}")
    print(f"   Total rows with embeddings: {valid_count:,}")
    print("=" * 70)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=str, default=None, help="Path to LPM .pt / .ckpt weights")
    args = parser.parse_args()
    main(checkpoint_path=args.checkpoint)

Writing extract_jump_latent_embeddings.py


In [ ]:
%%bash
/content/cpa_env/bin/python extract_jump_latent_embeddings.py

🚀 STARTING JUMP-CP LATENT PERTURBATION EXTRACTION PIPELINE
📥 Downloading gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/annotations_from_Roman/JUMP_CP_compound_metadata_with_SMILES_standard.csv -> /content/tmp_perturbridge/jump_input.csv...
   Size: 28.81 MB
📊 Loaded 115,796 JUMP-CP records.
🧪 Featurizing structures from 'SMILES_standard' using RDKit...
   Successfully featurized 115,689 / 115,796 (99.91%) molecules.
🧠 Computing latent embeddings on device: cpu...
ℹ️ Checkpoint path not provided or not found. Initializing base LPM compound encoder.
💾 Saving Parquet table locally: /content/tmp_perturbridge/jump_cp_latent_perturbation_embeddings.parquet...
📤 Uploading /content/tmp_perturbridge/jump_cp_latent_perturbation_embeddings.parquet -> gs://calico-reference-embeddings/embeddings/ChemPerturb-Bridge/latent_perturbation_embeddings/jump_cp_latent_perturbation_embeddings.parquet...
   Upload complete! Final GCS Size: 193.17 MB
✅ PIPELINE COMPLETED SUCCESS

/content/cpa_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
Morgan Fingerprinting: 100%|██████████| 115796/115796 [02:21<00:00, 819.71it/s]
